# SageMaker AI MLflow — Dataset Curation & Fine-Tuning a Smaller Model

In this notebook, you will curate a high-quality dataset from production traces captured by the Qwen3-8B Medical Triage Agent, and use it to fine-tune a smaller **Qwen2.5-7B** model via Supervised Fine-Tuning (SFT). This is a practical example of **knowledge distillation** — transferring the capabilities of a large model into a smaller, more cost-efficient one.

**What you will learn:**
- Load and filter MLflow traces from the online evaluation experiment using evaluation scores
- Prepare an SFT dataset in `prompt`/`completion` format from curated traces
- Register the dataset in the SageMaker AI Registry using `DataSet.create()`
- Run serverless fine-tuning with `SFTTrainer` (LoRA) on Qwen2.5-7B Instruct
- Deploy the fine-tuned model with DJL LMI + vLLM on a SageMaker endpoint
- Test the fine-tuned 7B model on medical triage queries

### Why Fine-Tune a Smaller Model?

| Aspect | Qwen3-8B (02-1) | Qwen2.5-7B Fine-Tuned (this notebook) |
|---|---|---|
| **Parameters** | 8 billion | 7 billion |
| **Instance** | ml.g5.2xlarge | ml.g5.2xlarge |
| **Cost** | Higher | ~4x lower |
| **Latency** | Higher | Lower |
| **Domain knowledge** | General + prompted | Specialized via SFT on curated traces |

By training on high-quality outputs from the larger model, the smaller model learns domain-specific patterns and can produce comparable results at a fraction of the cost.

### Prerequisites
- Completed the online evaluation notebook (02-2) with traces in the `medical-triage-agent` experiment
- A SageMaker AI MLflow App
- Service quota for serverless fine-tuning and `ml.g5.2xlarge` for deployment

## 1. Environment Setup

This module uses the **SageMaker Python SDK v3** (`sagemaker>=3.5.0`) which provides the `SFTTrainer`, `DataSet` registry, and new resource APIs. This is different from the v2 SDK used in earlier modules.

<div style="padding: 15px; background-color: #fff3cd; border-left: 5px solid #ffc107; color: #856404;">
<strong>⚠️ Important:</strong> The cell below installs libraries and restarts the kernel. After the restart, continue with the next cell.
</div>

In [ ]:
# Install SageMaker SDK v3 and dependencies
%pip install "sagemaker>=3.5.0" mlflow==3.9.0 sagemaker-mlflow==0.2.0 datasets==4.5.0 --quiet

# restart kernel
import IPython
IPython.Application.instance().kernel.do_shutdown(True) #automatically restarts kernel

In [ ]:
import json
import os
import boto3
import mlflow
import pandas as pd
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
bucket_name = sess.default_bucket()
default_prefix = sess.default_bucket_prefix
region = sess.boto_region_name
s3_client = boto3.client("s3")

try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

print(f"Role: {role}")
print(f"Bucket: {bucket_name}")
print(f"Region: {region}")

## 2. Configure SageMaker AI MLflow

Connect to your SageMaker AI MLflow App. We will read traces from the online evaluation experiment and create a new experiment for the fine-tuning run.

In [ ]:
# Retrieve values stored from previous labs
%store -r

%store
if MLFLOW_TRACKING_URI is None or MLFLOW_TRACKING_URI == '':
    print("MLFLOW_TRACKING_URI not found. Please enter your MLflow App ARN.")
MLFLOW_TRACKING_URI

In [ ]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

# Source experiment (all workshop traces)
source_experiment_name = "medical-triage-agent"

# New experiment for fine-tuning tracking
ft_experiment_name = "medical-triage-fine-tuning"
mlflow.set_experiment(ft_experiment_name)

print(f"MLflow tracking server: {mlflow.get_tracking_uri()}")
print(f"Source experiment: {source_experiment_name}")
print(f"Fine-tuning experiment: {ft_experiment_name}")

## 3. Load and Filter Traces from MLflow

We load all traces from the online evaluation experiment. Each trace contains:
- **Inputs**: The user's medical triage query
- **Outputs**: The Qwen3-8B model's response
- **Assessments**: Evaluation scores from the online evaluation scorers (Safety, RelevanceToQuery, etc.)

We filter to keep only high-quality traces where the model produced safe, relevant responses. These curated traces become the training data for the smaller model.

In [ ]:
# Load all traces from the online evaluation experiment
mlflow.set_experiment(source_experiment_name)
all_traces = mlflow.search_traces()

print(f"Total traces in '{source_experiment_name}': {len(all_traces)}")
all_traces.head()

### Inspect Available Columns

Let's see what columns are available in the traces DataFrame to understand the evaluation results structure.

In [ ]:
print("Available columns:")
for col in all_traces.columns:
    print(f"  {col}")

### Filter High-Quality Traces

We filter traces based on evaluation scores. Currently, we check for **safety** and **relevance** — only traces that passed these checks are included in the training dataset.

> **Tip:** To further refine the dataset, you can extend the filter logic below to exclude traces with negative human feedback (thumbs down) or poor scores on other metrics like coherence, professional tone, or bias. This is where you would add those additional filter conditions.

> **Note:** The exact column names for assessment results depend on how MLflow stores them. Adjust the filter conditions below based on the columns you see above. If assessment columns are not directly available, we will use all traces with valid inputs and outputs.

In [ ]:
# Extract prompt and completion from trace inputs/outputs
# Filter: only include traces that passed safety and relevance checks
curated_records = []
skipped_quality = 0

def check_assessment(assessments, name):
    """Check if a specific assessment passed (value='yes'). Returns True if passed or not found."""
    if not assessments:
        return True  # No assessments = include by default
    for a in assessments:
        if a.get('assessment_name') == name:
            feedback = a.get('feedback', {})
            return str(feedback.get('value', 'yes')).lower() != 'no'
    return True  # Assessment not found = include by default

for idx, trace in all_traces.iterrows():
    try:
        # Check safety and relevance assessments
        assessments = trace.get('assessments', [])
        if not check_assessment(assessments, 'safety') or not check_assessment(assessments, 'relevance_to_query'):
            skipped_quality += 1
            continue

        # Extract input (user query)
        inputs = trace.get("request")
        outputs = trace.get("response")
        
        if inputs is None or outputs is None:
            continue
        
        # Parse inputs — may be a string or dict
        if isinstance(inputs, str):
            try:
                inputs = json.loads(inputs)
            except json.JSONDecodeError:
                pass
        
        # Parse outputs — may be a string or dict
        if isinstance(outputs, str):
            try:
                outputs = json.loads(outputs)
            except json.JSONDecodeError:
                pass
        
        # Extract the prompt text
        if isinstance(inputs, dict):
            prompt = inputs.get("prompt", str(inputs))
        else:
            prompt = str(inputs)
        
        # Extract the completion text
        if isinstance(outputs, dict):
            completion = outputs.get("response", str(outputs))
        else:
            completion = str(outputs)
        
        # Skip empty or very short responses
        if len(completion.strip()) < 20:
            continue
        
        # Remove /no_think suffix from prompts if present
        prompt = prompt.replace(" /no_think", "").strip()
        
        curated_records.append({
            "prompt": prompt,
            "completion": completion,
        })
    except Exception as e:
        print(f"Skipping trace {idx}: {e}")
        continue

print(f"Curated {len(curated_records)} records from {len(all_traces)} traces")
print(f"Skipped {skipped_quality} traces that failed safety or relevance checks")

In [ ]:
# Preview a sample record
if curated_records:
    sample = curated_records[0]
    print(f"Sample prompt:\n{sample['prompt'][:300]}")
    print(f"\nSample completion:\n{sample['completion'][:300]}")

In [ ]:
all_records = curated_records
print(f"Total dataset size: {len(all_records)} records from production traces")

## 4. Prepare SFT Dataset

Convert the curated records into the SFT format expected by the SageMaker `SFTTrainer`. The trainer expects JSONL files with `prompt` and `completion` fields.

We split the data into training (80%) and validation (20%) sets.

In [ ]:
from sklearn.model_selection import train_test_split

# Split into train and validation
train_records, val_records = train_test_split(all_records, test_size=0.2, random_state=42)

print(f"Training samples: {len(train_records)}")
print(f"Validation samples: {len(val_records)}")

In [ ]:
# Save as JSONL files locally
os.makedirs("./sft_data/train", exist_ok=True)
os.makedirs("./sft_data/val", exist_ok=True)

def save_jsonl(records, filepath):
    with open(filepath, "w") as f:
        for record in records:
            f.write(json.dumps(record) + "\n")

save_jsonl(train_records, "./sft_data/train/dataset.jsonl")
save_jsonl(val_records, "./sft_data/val/dataset.jsonl")

print("Saved JSONL files:")
print(f"  Train: ./sft_data/train/dataset.jsonl ({len(train_records)} records)")
print(f"  Val:   ./sft_data/val/dataset.jsonl ({len(val_records)} records)")

In [ ]:
# Upload to S3
if default_prefix:
    s3_dataset_prefix = f"{default_prefix}/datasets/medical-triage-sft"
else:
    s3_dataset_prefix = "datasets/medical-triage-sft"

train_s3_path = f"s3://{bucket_name}/{s3_dataset_prefix}/train/dataset.jsonl"
val_s3_path = f"s3://{bucket_name}/{s3_dataset_prefix}/val/dataset.jsonl"

s3_client.upload_file("./sft_data/train/dataset.jsonl", bucket_name, f"{s3_dataset_prefix}/train/dataset.jsonl")
s3_client.upload_file("./sft_data/val/dataset.jsonl", bucket_name, f"{s3_dataset_prefix}/val/dataset.jsonl")

print(f"Uploaded to S3:")
print(f"  Train: {train_s3_path}")
print(f"  Val:   {val_s3_path}")

In [ ]:
# Clean up local files
import shutil
shutil.rmtree("./sft_data")
print("Local files cleaned up.")

## 5. Register Dataset in SageMaker AI Registry

The SageMaker AI Registry provides a centralized catalog for datasets, models, and evaluators. Registering the dataset makes it discoverable and reusable across training jobs.

We use `DataSet.create()` with `CustomizationTechnique.SFT` to register the training and validation datasets.

In [ ]:
from sagemaker.ai_registry.dataset import DataSet
from sagemaker.ai_registry.dataset_utils import CustomizationTechnique

dataset_train = DataSet.create(
    name="medical-triage-sft-train",
    source=train_s3_path,
    customization_technique=CustomizationTechnique.SFT,
    wait=True,
)
print(f"Training dataset ARN: {dataset_train.arn}")

dataset_val = DataSet.create(
    name="medical-triage-sft-val",
    source=val_s3_path,
    customization_technique=CustomizationTechnique.SFT,
    wait=True,
)
print(f"Validation dataset ARN: {dataset_val.arn}")

### Log Dataset to MLflow

We also log the dataset metadata to the MLflow experiment for full lineage tracking.

In [ ]:
with mlflow.start_run(run_name="dataset-curation"):
    mlflow.log_param("source_experiment", source_experiment_name)
    mlflow.log_param("total_traces", len(all_traces))
    mlflow.log_param("curated_from_traces", len(curated_records))
    mlflow.log_param("train_samples", len(train_records))
    mlflow.log_param("val_samples", len(val_records))
    mlflow.log_param("train_s3_path", train_s3_path)
    mlflow.log_param("val_s3_path", val_s3_path)
    mlflow.log_param("train_dataset_arn", dataset_train.arn)
    mlflow.log_param("val_dataset_arn", dataset_val.arn)
    mlflow.set_tag("stage", "dataset-curation")
    mlflow.set_tag("teacher_model", "qwen3-8b")

print("Dataset metadata logged to MLflow.")

## 6. Serverless Fine-Tuning with SFTTrainer

We use the SageMaker `SFTTrainer` to run serverless Supervised Fine-Tuning with LoRA on **Qwen2.5-7B Instruct**. This is a serverless job — no training infrastructure to manage.

The trainer:
1. Loads the base model from JumpStart
2. Applies LoRA adapters for parameter-efficient fine-tuning
3. Trains on the curated dataset
4. Merges the LoRA weights and registers the model in a Model Package Group
5. Tracks metrics in MLflow automatically

In [ ]:
base_model_id = "huggingface-llm-qwen2-5-7b-instruct"

if default_prefix:
    output_path = f"s3://{bucket_name}/{default_prefix}/{base_model_id}"
else:
    output_path = f"s3://{bucket_name}/{base_model_id}"

# Set MLflow endpoint for training job integration
os.environ["SAGEMAKER_MLFLOW_CUSTOM_ENDPOINT"] = (
    f"https://mlflow.sagemaker.{region}.app.aws"
)

print(f"Base model: {base_model_id}")
print(f"Output path: {output_path}")

### Create Model Package Group

A Model Package Group organizes versioned model artifacts from fine-tuning runs.

In [ ]:
from botocore.exceptions import ClientError
from sagemaker.core.resources import ModelPackageGroup

model_package_group_name = f"{base_model_id}-medical-triage-mpg"

try:
    model_package_group = ModelPackageGroup.get(
        model_package_group_name=model_package_group_name
    )
    print(f"Model Package Group already exists: {model_package_group_name}")
except ClientError:
    model_package_group = ModelPackageGroup.create(
        model_package_group_name=model_package_group_name,
        model_package_group_description="Medical triage SFT models fine-tuned from Qwen3-8B curated traces",
    )
    print(f"Created Model Package Group: {model_package_group_name}")

### Configure and Launch SFT Training Job

In [ ]:
from sagemaker.train.common import TrainingType
from sagemaker.train.sft_trainer import SFTTrainer

trainer = SFTTrainer(
    model=base_model_id,
    training_type=TrainingType.LORA,
    model_package_group=model_package_group_name,
    training_dataset=dataset_train,
    validation_dataset=dataset_val,
    s3_output_path=output_path,
    sagemaker_session=sess,
    role=role,
    accept_eula=True,
)

print("SFTTrainer configured.")

In [ ]:
# View default hyperparameters
print("Default hyperparameters:")
for k, v in trainer.hyperparameters.to_dict().items():
    print(f"  {k}: {v}")

In [ ]:
# Override hyperparameters for our small dataset
trainer.hyperparameters.learning_rate = 0.0001
trainer.hyperparameters.global_batch_size = 8
trainer.hyperparameters.max_epochs = 3
trainer.hyperparameters.lr_warmup_ratio = 0.1

print("Modified hyperparameters:")
for k, v in trainer.hyperparameters.to_dict().items():
    print(f"  {k}: {v}")

<div style="padding: 15px; background-color: #d1ecf1; border-left: 5px solid #17a2b8; color: #0c5460;">
<strong>ℹ️ Note:</strong> The serverless fine-tuning job typically takes 15-30 minutes depending on dataset size. Training metrics are automatically tracked in MLflow.
</div>

In [ ]:
# Launch the training job
training_job = trainer.train(wait=True)

TRAINING_JOB_NAME = training_job.training_job_name
print(f"Training job completed: {TRAINING_JOB_NAME}")

## 7. Deploy the Fine-Tuned Model

We deploy the fine-tuned Qwen2.5-7B model using DJL LMI with vLLM on a `ml.g5.2xlarge` instance — a much smaller (and cheaper) instance than the one used for the 8B teacher model.

### Get the Merged Model Artifact

In [ ]:
from sagemaker.core.resources import ModelPackage, ModelPackageGroup
from sagemaker.core import s3

model_package_version = "1"

model_package_group = ModelPackageGroup.get(model_package_group_name)
fine_tuned_model_package_arn = (
    f"{model_package_group.model_package_group_arn.replace('model-package-group', 'model-package', 1)}/{model_package_version}"
)

print(f"Model Package ARN: {fine_tuned_model_package_arn}")

model_package = ModelPackage.get(fine_tuned_model_package_arn)

# Get the merged model artifact S3 URI
merged_model_s3_uri = (
    s3.s3_path_join(
        model_package.inference_specification.containers[
            0
        ].model_data_source.s3_data_source.s3_uri,
        "checkpoints",
        "hf_merged",
    )
    + "/"
)
print(f"Merged model S3 URI: {merged_model_s3_uri}")

### Create Endpoint and Deploy

In [ ]:
model_name = f"{base_model_id}-medical-triage-sft"
endpoint_config_name = f"{base_model_id}-medical-triage-config"
endpoint_name = "workshop-qwen25-7b-sft"

instance_type = "ml.g5.2xlarge"
health_check_timeout = 700

print(f"Endpoint name: {endpoint_name}")
print(f"Instance type: {instance_type}")

In [ ]:
# Get the DJL LMI inference container image
CONTAINER_VERSION = "0.36.0-lmi18.0.0-cu128"
inference_image = f"763104351884.dkr.ecr.{region}.amazonaws.com/djl-inference:{CONTAINER_VERSION}"

env = {
    "HF_MODEL_ID": "/opt/ml/model",
    "OPTION_TRUST_REMOTE_CODE": "true",
    "OPTION_MODEL_LOADING_TIMEOUT": "3600",
    "OPTION_TENSOR_PARALLEL_DEGREE": "max",
    "SERVING_FAIL_FAST": "true",
    "OPTION_ROLLING_BATCH": "disable",
    "OPTION_ASYNC_MODE": "true",
    "OPTION_ENTRYPOINT": "djl_python.lmi_vllm.vllm_async_service",
    "OPTION_DTYPE": "bf16",
    "OPTION_QUANTIZE": "fp8",
    "OPTION_MAX_MODEL_LEN": json.dumps(1024 * 8),
}

print(f"Inference image: {inference_image}")

In [ ]:
from sagemaker.core.resources import Model
from sagemaker.core.shapes import (
    ContainerDefinition,
    ModelDataSource,
    S3ModelDataSource,
)

fine_tuned_model = Model.create(
    model_name=model_name,
    primary_container=ContainerDefinition(
        image=inference_image,
        model_data_source=ModelDataSource(
            s3_data_source=S3ModelDataSource(
                s3_uri=merged_model_s3_uri,
                s3_data_type="S3Prefix",
                compression_type="None",
            )
        ),
        environment=env,
    ),
    execution_role_arn=role,
)

print(f"Model created: {model_name}")

In [ ]:
from sagemaker.core.resources import Endpoint, EndpointConfig
from sagemaker.core.shapes import ProductionVariant

print(f"Creating EndpointConfig: {endpoint_config_name}")
endpoint_config = EndpointConfig.create(
    endpoint_config_name=endpoint_config_name,
    production_variants=[
        ProductionVariant(
            variant_name="AllTraffic",
            model_name=model_name,
            instance_type=instance_type,
            initial_instance_count=1,
            model_data_download_timeout_in_seconds=health_check_timeout,
        )
    ],
)

print(f"Creating Endpoint: {endpoint_name}")
endpoint = Endpoint.create(
    endpoint_name=endpoint_name, endpoint_config_name=endpoint_config_name
)
endpoint.wait_for_status("InService")
print(f"Endpoint {endpoint_name} is InService")

## 8. Test the Fine-Tuned Model

Let's test the fine-tuned 7B model with the same medical triage queries we used with the 32B model. The fine-tuned model should produce comparable results despite being ~4x smaller.

In [ ]:
import io

sagemaker_runtime = boto3.client("sagemaker-runtime")

class LineIterator:
    def __init__(self, stream):
        self.byte_iterator = iter(stream)
        self.buffer = io.BytesIO()
        self.read_pos = 0

    def __iter__(self):
        return self

    def __next__(self):
        while True:
            self.buffer.seek(self.read_pos)
            line = self.buffer.readline()
            if line and line[-1] == ord("\n"):
                self.read_pos += len(line)
                return line[:-1]
            try:
                chunk = next(self.byte_iterator)
            except StopIteration:
                if self.read_pos < self.buffer.getbuffer().nbytes:
                    continue
                raise
            if "PayloadPart" not in chunk:
                continue
            self.buffer.seek(0, io.SEEK_END)
            self.buffer.write(chunk["PayloadPart"]["Bytes"])

def parse_streaming_response(line_str):
    if not line_str.strip() or line_str.strip() == "data: [DONE]":
        return None
    if line_str.startswith("data: "):
        line_str = line_str[6:]
    try:
        data = json.loads(line_str)
        if "choices" in data:
            for choice in data["choices"]:
                if "delta" in choice and "content" in choice["delta"]:
                    return choice["delta"]["content"]
    except json.JSONDecodeError:
        pass
    return None

def invoke_fine_tuned_model(prompt):
    """Invoke the fine-tuned model with streaming."""
    request_body = {
        "messages": [
            {"role": "user", "content": [{"type": "text", "text": prompt}]}
        ],
        "max_tokens": 1024,
        "temperature": 0.3,
        "top_p": 0.9,
        "stop": ["<|im_end|>"],
        "stream": True,
    }
    response = sagemaker_runtime.invoke_endpoint_with_response_stream(
        EndpointName=endpoint_name,
        Body=json.dumps(request_body),
        ContentType="application/json",
    )
    generated_text = ""
    for line in LineIterator(response["Body"]):
        if line:
            content = parse_streaming_response(line.decode("utf-8"))
            if content:
                generated_text += content
                print(content, end="", flush=True)
    print()
    return generated_text

print("Inference utilities ready.")

In [ ]:
# Test 1: Emergency case
print("=" * 60)
print("Test 1: Emergency case — chest pain")
print("=" * 60)
response_1 = invoke_fine_tuned_model(
    "Assess the following patient and provide the likely condition, triage level, and treatment protocol.\n\n"
    "Patient ID: PT-001\nAge: 62\nSymptoms: chest pain, shortness of breath, sweating\nReported Severity: high"
)

In [ ]:
# Test 2: Non-urgent case
print("=" * 60)
print("Test 2: Non-urgent case — seasonal allergies")
print("=" * 60)
response_2 = invoke_fine_tuned_model(
    "Assess the following patient and provide the likely condition, triage level, and treatment protocol.\n\n"
    "Patient ID: PT-007\nAge: 29\nSymptoms: sneezing, runny nose, itchy eyes\nReported Severity: low"
)

In [ ]:
# Test 3: Urgent case
print("=" * 60)
print("Test 3: Urgent case — asthma attack")
print("=" * 60)
response_3 = invoke_fine_tuned_model(
    "Assess the following patient and provide the likely condition, triage level, and treatment protocol.\n\n"
    "Patient ID: PT-005\nAge: 8\nSymptoms: wheezing, difficulty breathing, coughing\nReported Severity: high"
)

## 9. Cleanup

Uncomment and run the cells below to delete the deployed resources when you are done.

> **Note:** Delete resources in order: Inference Component → Model → Endpoint → Endpoint Config.

In [ ]:
# Delete inference component
# InferenceComponent.get(inference_component_name=ic_name).delete()
# print(f"Deleted InferenceComponent: {ic_name}")

In [ ]:
# Delete model
# Model.get(model_name=model_name).delete()
# print(f"Deleted Model: {model_name}")

In [ ]:
# Delete endpoint
# Endpoint.get(endpoint_name=endpoint_name).delete()
# print(f"Deleted Endpoint: {endpoint_name}")

In [ ]:
# Delete endpoint config
# EndpointConfig.get(endpoint_config_name=endpoint_config_name).delete()
# print(f"Deleted EndpointConfig: {endpoint_config_name}")

In [ ]:
# Also delete the teacher model endpoint if no longer needed
# import sagemaker as sagemaker_v1
# teacher_predictor = sagemaker_v1.Predictor(endpoint_name="workshop-qwen3-8b", sagemaker_session=sagemaker_v1.Session())
# teacher_predictor.delete_endpoint()
# print("Deleted teacher endpoint: workshop-qwen3-8b")